# Suite VLGPRT — Printing Verilog

The printer writes one fixed layout and the parentheses precedence needs; printing validates the tree and checks
the standard first.

In [ ]:
import { Errors, type Syntax as F } from "@mbse/programs/Framework";
import { SystemVerilog2023, Syntax as S, Verilog2005 } from "@mbse/programs/Verilog";

import { equal, raises } from "./support.js";

const P = (node: F.SyntaxNode) => SystemVerilog2023.print(node);
const name = (spelling: string) => new S.Identifier({ spelling });
const ref = (spelling: string) => new S.NameExpression({ name: name(spelling) });
const number = (spelling = "1") => new S.IntegerLiteral({ spelling });
const binary = (left: any, operator: string, right: any) => new S.BinaryExpression({ left, operator, right });
const [a, b, c] = [ref("a"), ref("b"), ref("c")];

/** Text reads and prints back to itself. */
function roundtrip(text: string): string {
  const printed = P(SystemVerilog2023.parse(text));
  equal(printed, text);
  return printed;
}

## VLGPRT-01 · Parentheses follow IEEE 1800's precedence, and right-associative operators

In [ ]:
equal(P(binary(binary(a, "+", b), "*", c)), "(a + b) * c");
equal(P(binary(a, "-", binary(b, "-", c))), "a - (b - c)");
equal(P(binary(binary(a, "-", b), "-", c)), "a - b - c");
equal(P(binary(a, "->", binary(b, "->", c))), "a -> b -> c");
equal(P(binary(binary(a, "->", b), "->", c)), "(a -> b) -> c");
const cond = () => new S.ConditionalExpression({ condition: ref("a"), consequence: ref("b"), alternative: ref("c") });
equal(P(new S.ConditionalExpression({ condition: cond(), consequence: ref("b"), alternative: cond() })),
  "(a ? b : c) ? b : a ? b : c");
equal(P(new S.UnaryExpression({ operator: "-", operand: binary(a, "+", b) })), "-(a + b)");
equal(P(new S.UnaryExpression({ operator: "~", operand: new S.UnaryExpression({ operator: "-", operand: a }) })), "~(-a)");
equal(P(new S.IndexExpression({ value: binary(a, "+", b), index: number() })), "(a + b)[1]");
equal(P(new S.InsideExpression({ value: binary(a, "==", b), set: [number()] })), "(a == b) inside {1}");
equal(P(binary(a, "+", new S.InsideExpression({ value: b, set: [c] }))), "a + (b inside {c})");
equal(P(binary(new S.AssignmentExpression({ target: a, operator: "=", value: b }), "+", c)), "(a = b) + c");
equal(P(new S.IncrementExpression({ operator: "++", operand: binary(a, "+", b) })), "++(a + b)");
equal(P(new S.CastExpression({ type: binary(a, "+", b), value: c })), "(a + b)'(c)");
console.log("ok");

## VLGPRT-02 · Every expression and part prints alone

In [ ]:
const cases: [F.SyntaxNode, string][] = [
  [new S.MemberExpression({ value: a, member: name("m") }), "a.m"],
  [new S.RangeSelect({ value: a, left: number("7"), operator: ":", right: number("0") }), "a[7:0]"],
  [new S.RangeSelect({ value: a, left: number("0"), operator: "+:", right: number("4") }), "a[0 +: 4]"],
  [new S.RealLiteral({ spelling: "1.5" }), "1.5"], [new S.TimeLiteral({ spelling: "1ns" }), "1ns"],
  [new S.UnbasedUnsizedLiteral({ value: "z" }), "'z"], [new S.StringLiteral({ text: "hi" }), '"hi"'],
  [new S.Concatenation({ items: [a, number()] }), "{a, 1}"], [new S.Replication({ count: number("2"), items: [a] }), "{2{a}}"],
  [new S.AssignmentPattern({ items: [new S.PatternItem({ value: number() }), new S.PatternItem({ key: a, value: number("2") }),
    number("3")] }), "'{default: 1, a: 2, 3}"],
  [new S.AssignmentPattern({ type: new S.NamedType({ name: name("t") }), items: [number()] }), "t'{1}"],
  [new S.CallExpression({ callee: a, arguments: [number(), new S.NamedConnection({ name: name("b"), value: number("2") })] }),
    "a(1, .b(2))"],
  [new S.SystemCall({ name: "$bits", arguments: [new S.IntegerAtomType({ keyword: "int" })] }), "$bits(int)"],
  [new S.SystemCall({ name: "$time" }), "$time"],
  [new S.CastExpression({ type: new S.ImplicitType({ signing: "unsigned" }), value: a }), "unsigned'(a)"],
  [new S.MacroUsage({ name: name("M"), arguments: "1, 2" }), "`M(1, 2)"], [new S.MacroUsage({ name: name("W") }), "`W"],
  [new S.DollarExpression(), "$"],
  [new S.NameExpression({ name: new S.ScopedName({ scope: name("p"), name: name("x") }) }), "p::x"],
  [new S.RangeDimension({ left: number("3"), right: number("0") }), "[3:0]"], [new S.UnsizedDimension(), "[]"],
  [new S.AssociativeDimension(), "[*]"], [new S.QueueDimension({ bound: number("4") }), "[$:4]"], [new S.QueueDimension(), "[$]"],
  [new S.IntegerVectorType({ keyword: "bit", signing: "signed", dimensions: [new S.SizeDimension({ size: number("4") })] }),
    "bit signed [4]"],
  [new S.ImplicitType({ dimensions: [new S.RangeDimension({ left: number("1"), right: number("0") })] }), "[1:0]"],
  [new S.EnumType({ base: new S.IntegerAtomType({ keyword: "int" }), members: [new S.EnumMember({ name: name("A"), value: number() })] }),
    "enum int {A = 1}"],
  [new S.StructType({ keyword: "union", members: [new S.StructMember({ type: new S.KeywordType({ keyword: "string" }),
    declarators: [new S.VariableDeclarator({ name: name("s") })] })] }), "union { string s; }"],
  [new S.AnsiPort({ name: name("p") }), "p"], [new S.InterfacePort({ name: name("i") }), "interface i"],
  [new S.PortReference({ name: name("r") }), "r"],
  [new S.IncludeDirective({ path: "types.svh", system: true }), "`include <types.svh>"],
  [new S.TimedStatement({ timing: new S.DelayControl({ value: binary(a, "+", number()) }) }), "#(a + 1);"],
  [new S.VirtualInterfaceType({ interface: name("bus"), parameters: [number("4")], modport: name("mp") }), "virtual bus #(4).mp"],
  [new S.NewExpression({ scope: new S.ScopedName({ scope: name("p"), name: new S.ParameterizedName({ name: name("c"),
    parameters: [number()] }) }), arguments: [a] }), "p::c #(1)::new(a)"],
  [new S.NewCopyExpression({ value: new S.CallExpression({ callee: a }) }), "new a()"],
  [new S.NewCopyExpression({ value: binary(a, "+", number()) }), "new (a + 1)"],
  [new S.ForwardTypedefDeclaration({ name: name("t") }), "typedef t;"],
];
for (const [node, text] of cases) equal(P(node), text);
console.log("ok");

## VLGPRT-03 · Layout: blocks open on their header's line, `end else`, one item per line

In [ ]:
roundtrip(`module m;
    always_ff @(posedge clk) begin
        if (a) begin
            b <= 1;
        end else if (c) begin
            b <= 2;
        end else begin
            b <= 3;
        end
        if (a)
            b <= 4;
        else if (c)
            b <= 5;
        else
            b <= 6;
        unique case (a) inside
            1, [2:3]: b <= 1;
            4: begin
                b <= 2;
            end
            default: ;
        endcase
        for (int i = 0; i < 2; i++)
            a = i;
        do begin
        end while (a);
        do
            a = 1;
        while (a);
        wait (a);
        wait (a) b = 1;
        wait (a) begin
        end
        #1 a = 1;
        @(posedge clk) begin
        end
        assert (a);
        assert (a) b = 1; else b = 2;
        assert (a) begin
        end else begin
        end
        assert (a) else $error("x");
    end
endmodule
`);
roundtrip(`module m;
    if (W) begin : a
    end else if (V) begin : b
    end else
        assign x = 1;

    if (W)
        assign x = 2;
    else
        assign x = 3;

    case (W)
        1:
            assign x = 4;
        default: begin
        end
    endcase

    for (genvar i = 0; i < 2; i++) begin : g
    end : g

    for (j = 0; j < 2; j = j + 1)
        assign x = 5;
endmodule
`);
console.log("ok");

## VLGPRT-04 · Layout: design units, declarations and directives, with blank lines around what spans lines

In [ ]:
roundtrip(`// lead
\`timescale 1ns / 1ps

/* the package */
package p;
    typedef struct packed {
        logic a;
        logic b;
    } s;
    var struct packed {
        logic a;
        logic b;
    } v;
    typedef union { int i; } u;

    function automatic int f(input int a = 1, output b);
        return a;
    endfunction : f

    task t;
        input a;
    endtask
endpackage : p

module m (a, b);
    input wire a;
    output reg b;

\`ifdef X
    wire x;
\`elsif Y
    wire y;
\`else
    wire z;
\`endif
endmodule

interface i
    import p::*;
#(
    parameter int W = 1,
    parameter type T = logic
) (
    input logic clk
);
    modport mp (input clk);
endinterface

program q #(
    localparam N = 1
);
endprogram
`);
const text = P(SystemVerilog2023.parse("// lead\n`timescale 1ns / 1ps\n/* the package */\npackage p; typedef struct packed { logic a; logic b; } s; endpackage\n"));
equal(text, "// lead\n`timescale 1ns / 1ps\n\n/* the package */\npackage p;\n    typedef struct packed {\n        logic a;\n        logic b;\n    } s;\nendpackage\n");
console.log("ok");

## VLGPRT-05 · Printing validates the tree and checks the standard

In [ ]:
raises(Errors.PrintError, () => P(new S.BinaryExpression({ operator: "+", right: number() })), "a BinaryExpression needs a left");
raises(Errors.PrintError, () => Verilog2005.print(new S.PackageDeclaration({ name: name("p") })),
  "PackageDeclaration is not Verilog, but this is Verilog-2005");
equal(Verilog2005.print(new S.ModuleDeclaration({ keyword: "module", name: name("m") })), "module m;\nendmodule");
equal(P(new S.Comment({ text: " c" })), "// c");
equal(P(new S.Comment({ block: true, text: " c " })), "/* c */");
console.log("ok");

## VLGPRT-06 · Layout: trailing comments, conditionals among statements, bodies that span lines

In [ ]:
roundtrip(`module m;  // trailing
    wire a;  /* after a */

    initial begin
        x = (a inside {1, 2}) && b;

\`ifdef A
        x = 1;
\`else
        x = 2;
\`endif

        case (a)
            1:
                if (b)
                    x = 1;
                else
                    x = 2;
            default: ;
        endcase
        assert (a) $display("pass");
        assert (a) begin
            x = 1;
        end else $error("fail");
        ;
    end
endmodule
`);
console.log("ok");

## VLGPRT-07 · Layout: every option of headers, ports, declarations and statements

In [ ]:
roundtrip(`\`define EMPTY

module a;
endmodule  // after a

// before b
module b
    import p::*;
(x, y);
    input var logic x;
    output y;
endmodule

module c
    import p::*;
(
    output var logic z = 1'b0,
    w
);
    wire #3 n;
    wire #(d + 1) m;
    const var static int k = 1;
    var v;
    static struct {
        logic a;
        logic b;
    } s;
    struct {
        logic a;
        logic b;
    } u;
    const struct packed {
        logic a;
        logic b;
    } [1:0] t = '0;
    typedef struct packed { logic a; } one;
    typedef enum {A} e;

    /* two
       lines */
    function int f(input var int i, j);
        return 1;
    endfunction

    task r;
        return;
    endtask

    initial begin
        ->> ev;
        disable named;
        disable fork;
        unique if (a)
            b = 1;
        for (; ; );
        for (int k = 0, m = 1; ; );
        #(a + 1) b = 1;
    end
endmodule
`);
console.log("ok");

## VLGPRT-08 · Layout: classes, their parameters, bases, members and prototypes

In [ ]:
roundtrip(`typedef class driver;

virtual class base #(
    type T = int,
    int N = 2
) extends parent #(T) implements i1, i2;
    local const static int count = 0;
    extern virtual function void f(int x);
    pure virtual task t;

    function new(int x = 0);
        super.new(x);
    endfunction
endclass : base

class derived extends base #(byte, 3)(1, .x(2));
endclass

interface class i1 extends j, k;
    pure virtual function void h;
endclass

module m #(
    int W = 8,
    type U = logic
);
    wire struct packed {
        logic a;
        logic b;
    } w;
endmodule
`);
console.log("ok");